# EO Specimen label parsing

In [ ]:
import re
import pandas as pd

# load specimen records associated with EOs as entered into Biotics -- see SpecimenQuery.txt file
Biotics_specimens = pd.read_csv("EO_Specimens_20250418.csv", encoding='latin1')

# search patterns for collection/catalog/accession number within 
# word starts with collection, then space or not, # sign, space or not, then any alphabetic characters or digits separated by - or . The second group allows for comma separated number, but requires digits in the second to avoid adding words endlessly.
collection_pattern = re.compile(
    r"(?:\bcollection\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

catalog_pattern = re.compile(
    r"(?:\bcatalog\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

accession_pattern = re.compile(
    r"(?:\baccession\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

#Example -- O'Kane, S. 1985. Specimen (collection #2103) at University of Colorado Herbarium.

#Search patterns for collector name and year.
collector_pattern = re.compile(r"^[A-Za-z,.\s]+")
year_pattern = re.compile(r"\b((?:18|19|20)\d{2})\b")



herbarium_df = pd.read_excel('HerbariumCodes.xlsx')

# Create a list of herbarium names and codes used in SEINet export
herbarium_dict = dict(zip(herbarium_df['textName'].astype(str), herbarium_df['institutionCode_CNHP']))

# Compile a regular expression pattern for each herbarium name
herbarium_patterns = {name: re.compile(r'\b' + name + r'.*', re.IGNORECASE) for name in herbarium_dict}

# Function to search for herbarium codes in text
def find_herbarium_code(text, patterns, herbarium_dict):
    for name, pattern in patterns.items():
        if pattern.search(text):
            return herbarium_dict[name]
    return None  # If no match is found

# check for collection number ranges
def parse_collection_range(entry):
    if '-' in entry:
        # Check if it's a range (e.g., 475-477)
        if len(entry.split('-')) == 2:
            start, end = entry.split('-')
            try:
                start_num = int(start)
                end_num = int(end)
                # Only expand if the range difference is 4 or less and over 100
                if abs(end_num - start_num) <= 4 and start_num > 100:
                    return list(range(start_num, end_num + 1))
                else:
                    return [entry]
            except ValueError:
                return [entry]
        else:
            return [entry]
    else:
        return [entry]

results = []
for index, row in Biotics_specimens.iterrows():
    text = row["SPECIMEN_DESC"]
    eo_id = row["EO_ID"]
    if year_pattern.search(text) is not None:
        year = year_pattern.search(text).group()
    herbarium = find_herbarium_code(text, herbarium_patterns, herbarium_dict)
    if collector_pattern.search(text) is not None:
        collector = collector_pattern.search(text).group()
    if collection_pattern.search(text) is not None:
        for num in collection_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, num, None, None])
                #print([eo_id, collector, year, herbarium, num, None, None])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, item, None, None])
    if catalog_pattern.search(text) is not None:
        for num in catalog_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, None, num, None])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, None, item, None])
    if accession_pattern.search(text) is not None:
        for num in accession_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, None, None, num])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, None, None, item])

column_names = ["EO_ID", "sourceText", "collector", "year", "institutionCode", "collectionNumber", "catalogNumber", "accessionNumber"]
result_df = pd.DataFrame(results, columns=column_names)

# merge parsed specimen labels with EO SNAME data
EO_df = pd.read_csv("EO_download_20250421.csv")

result_df = result_df.merge(EO_df, how="left", on="EO_ID")
#result_df = result_df[["EO_ID", "SName", "ElCode", "sourceText", "collector", "year", "institutionCode", "collectionNumber", "catalogNumber", "accessionNumber", "Precision", "RepAcc"]]
# filter to only vascular plant specimen labels
#result_df = result_df[result_df["ElCode"].str.startswith('P', na=False)]
result_df.to_csv("EO_Specimens_parsed_20250418.csv", index=False)


# EO specimen label matching to SEINet

In [ ]:
import pandas as pd
import re
import numpy as np
# use combination of collector, year, and collection number to identify specimens in Biotics (return those with different SNAMEs to be reviewed manually for updates)
# include two columns in SEINet spreadsheet, one for Matched: Y, S, H; (yes, species discrepancy, herbarium discrepancy) and one for EOID 

EO_specimens = pd.read_csv("EO_Specimens_parsed_20250418.csv")
SEINet = pd.read_csv(r"C:\Users\chollenb\OneDrive - Colostate\Documents\Data\SEINet\SEINet_translated_01152025.csv")

# Add a new column to SEINet to track matched rows in EO specimen sheet
SEINet["matched"] = None
SEINet["EO_ID_match"] = None


# handle collection/catalog/accession numbers
number_columns = ['collectionNumber', 'catalogNumber', 'accessionNumber']
# strip leading zeros from SEINet number columns to match the EO labels in case either one got zeros removed, also remove ".0" from those that were stored as floats.
for column in number_columns:
    SEINet[column]=SEINet[column].astype(str).str.lstrip('0')
    SEINet[column] = SEINet[column].str.replace(r'\.0$', '', regex=True)
    SEINet[column] = SEINet[column].replace("nan", np.nan)
    EO_specimens[column]=EO_specimens[column].astype(str).str.lstrip('0')
    EO_specimens[column] = EO_specimens[column].str.replace(r'\.0$', '', regex=True)
    EO_specimens[column] = EO_specimens[column].replace("nan", np.nan)

EO_specimens["SEINet_id"]=None
EO_specimens["SEINet_links"] = None
EO_specimens["SEINet_links_all"] = None
EO_specimens["match_status"]=None
EO_specimens["year_check"]=None
EO_specimens["collector_check"]=None

##########################################################
# Loop through EO specimens
for index, row in EO_specimens.iterrows():

    #############################################
    # check year, collector name, and collection number

    # Filter SEINet by year column and check if any records
    matched_rows = SEINet[SEINet["year"]==row["year"]]
    if not matched_rows.empty:
        year_check="Y"
    else:
        print("fail")
        print(row["year"])
        year_check="N"
    # take the last name from the collectors and check collector name match in SEINet
    last_name = re.split(r'[,.\s]+', row['collector'])[0].strip()
    matched_rows = matched_rows[matched_rows['collector'].str.contains(last_name, case=False, na=False)]
    SEINet_ids=None
    SEINet_ids_all=None
    match_status="N"
    if not matched_rows.empty:
        collector_check="Y"
    else:
        collector_check="N"
        collector_check=f"Lastname: {last_name}, Collector: {row['collector']}"
        EO_specimens.at[index, "SEINet_id"] = SEINet_ids
        EO_specimens.at[index, "match_status"] = match_status
        EO_specimens.at[index, "year_check"] = year_check
        EO_specimens.at[index, "collector_check"] = collector_check

    #  Identify the first non-NaN column in number_columns for the row
    non_na_col = next((col for col in number_columns if not pd.isna(row[col])), None)
    # If a non-NaN number_columns exists, further filter matched_rows by this column
    if non_na_col:
        #print(row['SName'])
        #print(row[non_na_col])
        #print(matched_rows[non_na_col])
        value = row[non_na_col]
        # allow for numbers incorrectly attributed in Biotics, ex: a collectionNumber called a catalogNumber
        matched_rows = matched_rows[(matched_rows['collectionNumber'] == value) | (matched_rows['catalogNumber'] == value) | (matched_rows['accessionNumber'] == value)]   
        # if there are any matches to year, collector last name, and collection/catalog/accession number
        if not matched_rows.empty:
            # mark those who made it to the species check
            SEINet.loc[matched_rows.index, "matched"] = "S"
            # update matching ids
            SEINet_ids_all = matched_rows["SEINet_id"]
            SEINet_links_all = matched_rows["references"]
            if len(SEINet_ids_all) == 1:
                SEINet_ids_all = SEINet_ids_all.iloc[0]  # Extract the single value
            else:
                SEINet_ids_all = "; ".join(SEINet_ids_all.astype(str))  # Join multiple IDs as a string
            if len(SEINet_links_all) == 1:
                SEINet_links_all = SEINet_links_all.iloc[0]  # Extract the single value
            else:
                SEINet_links_all = "; ".join(SEINet_links_all.astype(str))  # Join multiple IDs as a string
            match_status = "S"
            # recording matching EO_ID or document overwrites
            if SEINet.loc[matched_rows.index, "EO_ID_match"] is not None:
                SEINet.loc[matched_rows.index, "EO_ID_match"] = row["EO_ID"]
            else:
                print("Caution: overwriting!")
                print(SEINet.loc[matched_rows.index])

            ##############################
            # check species match
            matched_rows = matched_rows[matched_rows["SNAME"] == row["SNAME"]]
            if not matched_rows.empty:
                # mark those who made it to the herbarium check
                SEINet.loc[matched_rows.index, "matched"] = "H"
                # update matching ids
                SEINet_ids = matched_rows["SEINet_id"]
                if len(SEINet_ids) == 1:
                    SEINet_ids = SEINet_ids.iloc[0]  # Extract the single value
                else:
                    SEINet_ids = "; ".join(SEINet_ids.astype(str))  # Join multiple IDs as a string
                match_status = "H"
                ###########################
                # check herbarium match
                matched_rows = matched_rows[matched_rows["institutionCode"] == row["institutionCode"]]
                if not matched_rows.empty:
                    # mark those complete matches
                    SEINet.loc[matched_rows.index, "matched"] = "Y"
                    # update matching ids
                    SEINet_ids = matched_rows["SEINet_id"]
                    SEINet_links = matched_rows["references"]
                    if len(SEINet_ids) == 1:
                        SEINet_ids = SEINet_ids.iloc[0]  # Extract the single value
                    else:
                        SEINet_ids = "; ".join(SEINet_ids.astype(str))  # Join multiple IDs as a string
                    if len(SEINet_links) == 1:
                        SEINet_links = SEINet_links.iloc[0]  # Extract the single value
                    else:
                        SEINet_links = "; ".join(SEINet_links.astype(str))  # Join multiple IDs as a string
                    match_status = "Y"
            EO_specimens.at[index, "SEINet_id"] = SEINet_ids
            EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
            EO_specimens.at[index, "SEINet_links"] = SEINet_links
            EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
            EO_specimens.at[index, "match_status"] = match_status
            EO_specimens.at[index, "year_check"] = year_check
            EO_specimens.at[index, "collector_check"] = collector_check
        else:
            EO_specimens.at[index, "SEINet_id"] = SEINet_ids
            EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
            EO_specimens.at[index, "SEINet_links"] = SEINet_links
            EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
            EO_specimens.at[index, "match_status"] = match_status
            EO_specimens.at[index, "year_check"] = year_check
            EO_specimens.at[index, "collector_check"] = collector_check
    else: 
        EO_specimens.at[index, "SEINet_id"] = None
        EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
        EO_specimens.at[index, "SEINet_links"] = SEINet_links
        EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
        EO_specimens.at[index, "match_status"] = "Specimen number missing!"
        EO_specimens.at[index, "year_check"] = year_check
        EO_specimens.at[index, "collector_check"] = collector_check
    
    

EO_specimens.to_csv("EO_specimens_parse_check_20250421.csv", index=False)


# process for removing duplicates in EO specimen labels where there were multiple collection number, so we know specimen labels that truly had no match
# Define valid match statuses
valid_statuses = {'Y', 'S', 'H'}
# Create a mask 
mask = EO_specimens.groupby('sourceText')['match_status'].transform(
    lambda x: any(status in valid_statuses for status in x)
) & (EO_specimens['match_status'] == 'N')

# Drop rows where the mask is True
EO_specimens_cleaned = EO_specimens[~mask]

# save match files to csv
EO_specimens_cleaned.to_csv("EO_specimens_parse_check_20250421_cleaned.csv", index=False)
SEINet.to_csv("SEINet_specimen_match_20250421.csv", index=False)


C:\Users\chollenb\AppData\Local\Temp\ipykernel_26244\1674875006.py:9: DtypeWarning: Columns (3,10,11,12,13,14,16,20,21,22,23,24,25,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  SEINet = pd.read_csv(r"C:\Users\chollenb\OneDrive - Colostate\Documents\Data\SEINet\SEINet_translated_01152025.csv")


In [ ]:
#####################################
# additional code to filter matched specimen table to include single result per EO specimen, in order of Y, H, S for matches.

# Your custom sort order
sort_order = ['Y', 'H', 'S']

df = pd.read_csv("EO_specimens_parse_check_20250421_cleaned.csv")
# Create a categorical column with custom order
df['match_status_cat'] = pd.Categorical(
    df['match_status'],
    categories=sort_order + sorted(set(df['match_status'].unique()) - set(sort_order)),
    ordered=True
)

# Sort by the categorical column
df_sorted = df.sort_values('match_status_cat').drop(columns='match_status_cat')

df_sorted = df_sorted.drop_duplicates(subset=["EO_ID", "COUNTY_NAME", "sourceText"], keep="first")

df_sorted.to_excel("EO_specimens_parse_check_20250421_cleanest.xlsx", index=False)